# KMP — Failure Function & Search

In [1]:
%config InlineBackend.figure_format = "retina"

import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

warnings.filterwarnings("ignore", message=".*tight_layout.*")


def safe_tight_layout(fig=None):
    """Call tight_layout suppressing the aspect-ratio incompatibility warning."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        if fig is None:
            plt.tight_layout()
        else:
            fig.tight_layout()


plt.rcParams.update({
    "font.family": "monospace",
    "font.size": 13,
    "figure.facecolor": "#fafafa",
    "figure.dpi": 150,
    "savefig.dpi": 150,
})

COLORS = {
    "match": "#4CAF50",
    "mismatch": "#F44336",
    "current": "#FFC107",
    "skip": "#90CAF9",
    "default": "#E0E0E0",
    "pattern": "#BBDEFB",
    "border": "#7E57C2",
}


def draw_string_row(ax, y, string, label="", highlights=None, offset=0):
    """Draw a row of character boxes at vertical position y."""
    highlights = highlights or {}
    for i, ch in enumerate(string):
        color = highlights.get(i, COLORS["default"])
        rect = mpatches.FancyBboxPatch(
            (i + offset, y), 0.9, 0.9,
            boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="#555", linewidth=1.2,
        )
        ax.add_patch(rect)
        ax.text(i + offset + 0.45, y + 0.45, ch,
                ha="center", va="center", fontsize=14, fontweight="bold")
    if label:
        ax.text(offset - 0.3, y + 0.45, label,
                ha="right", va="center", fontsize=12, color="#555")


def make_stepper(step_widget, label="Step"):
    """Create a prev/next/first/last button bar tied to an IntSlider (hidden)."""
    btn_first = widgets.Button(description="|<", layout=widgets.Layout(width="40px"))
    btn_prev  = widgets.Button(description="<", layout=widgets.Layout(width="40px"))
    btn_next  = widgets.Button(description=">", layout=widgets.Layout(width="40px"))
    btn_last  = widgets.Button(description=">|", layout=widgets.Layout(width="40px"))
    counter   = widgets.Label(value=f"{label}: {step_widget.value}/{step_widget.max}")

    def update_label(*_):
        counter.value = f"{label}: {step_widget.value}/{step_widget.max}"

    step_widget.observe(update_label, "value")
    step_widget.observe(update_label, "max")

    def on_first(_): step_widget.value = step_widget.min
    def on_prev(_):  step_widget.value = max(step_widget.min, step_widget.value - 1)
    def on_next(_):  step_widget.value = min(step_widget.max, step_widget.value + 1)
    def on_last(_):  step_widget.value = step_widget.max

    btn_first.on_click(on_first)
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    btn_last.on_click(on_last)

    return widgets.HBox([btn_first, btn_prev, counter, btn_next, btn_last])


print("Helpers loaded (retina mode).")

Helpers loaded (retina mode).


## KMP — Failure Function (sp values) & Search

### sp (failure function) — Step by Step

Step through the computation of `sp[j]` = length of the longest proper border of `P[0..j]`. At each step you see:
- The **comparison** `P[k] vs P[j]`
- Any **fallbacks** through the sp chain
- The **prefix/suffix border** highlighted on the pattern

In [2]:
def compute_sp(P):
    """Compute sp[j] = length of longest proper border of P[0..j]."""
    m = len(P)
    sp = [0] * m
    k = 0
    for j in range(1, m):
        while k > 0 and P[k] != P[j]:
            k = sp[k - 1]
        if P[k] == P[j]:
            k += 1
        sp[j] = k
    return sp


def sp_trace(P):
    """Step-by-step sp computation. Each snapshot captures full state."""
    m = len(P)
    sp = [0] * m
    k = 0
    snapshots = []
    for j in range(1, m):
        fallbacks = []
        while k > 0 and P[k] != P[j]:
            old_k = k
            k = sp[k - 1]
            fallbacks.append((old_k, k))
        matched = P[k] == P[j]
        if matched:
            k += 1
        sp[j] = k
        snapshots.append({
            "j": j, "k_before": k - 1 if matched else k,
            "k_after": k, "matched": matched,
            "sp": list(sp), "fallbacks": fallbacks,
            "border_len": sp[j],
        })
    return snapshots


def draw_sp_step(P, step_idx):
    snapshots = sp_trace(P)
    if not snapshots:
        return
    step_idx = min(step_idx, len(snapshots) - 1)
    snap = snapshots[step_idx]
    m = len(P)
    j = snap["j"]
    sp = snap["sp"]
    border_len = snap["border_len"]
    matched = snap["matched"]
    k_cmp = snap["k_before"]

    fig, axes = plt.subplots(2, 1, figsize=(max(m * 0.85, 7), 5.0),
                              gridspec_kw={"height_ratios": [3, 2]})

    # ── Panel 1: Pattern with border highlights ──
    ax = axes[0]
    ax.set_xlim(-2.0, m + 1.0)
    ax.set_ylim(-1.8, 2.8)
    ax.set_aspect("equal")
    ax.axis("off")

    # P row highlights
    p_hi = {}
    if border_len > 0:
        for i in range(border_len):
            p_hi[i] = COLORS["match"]
        for i in range(j - border_len + 1, j + 1):
            p_hi[i] = COLORS["skip"]
    if matched:
        p_hi[k_cmp] = COLORS["match"]
        p_hi[j] = COLORS["match"]
    else:
        p_hi[j] = COLORS["mismatch"]

    draw_string_row(ax, 0.8, P, label="P", highlights=p_hi)

    # dim unprocessed positions
    for i in range(j + 1, m):
        rect = mpatches.FancyBboxPatch(
            (i, 0.8), 0.9, 0.9, boxstyle="round,pad=0.05",
            facecolor="#F0F0F0", edgecolor="#CCC", linewidth=0.8)
        ax.add_patch(rect)
        ax.text(i + 0.45, 1.25, P[i], ha="center", va="center",
                fontsize=14, fontweight="bold", color="#BBB")

    # index row above
    for i in range(m):
        ax.text(i + 0.45, 2.0, str(i), ha="center", fontsize=8, color="#999")

    # j pointer above the box
    ax.annotate(f"j={j}", xy=(j + 0.45, 1.7), xytext=(j + 0.45, 2.4),
                fontsize=9, ha="center", color="#C62828", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#C62828", lw=1.5))

    # k pointer below the box
    ax.annotate(f"k={k_cmp}", xy=(k_cmp + 0.45, 0.8), xytext=(k_cmp + 0.45, 0.2),
                fontsize=9, ha="center", color="#1565C0", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#1565C0", lw=1.5))

    # comparison text — placed to the right of the pattern, not between k and j
    cmp_color = COLORS["match"] if matched else COLORS["mismatch"]
    cmp_sym = "==" if matched else "!="
    cmp_text = f"P[{k_cmp}]='{P[k_cmp]}' {cmp_sym} P[{j}]='{P[j]}'"
    ax.text(m + 0.5, 1.25, cmp_text, fontsize=9, va="center", color=cmp_color, fontweight="bold")

    # border brackets below
    if border_len > 0:
        y_brk = -0.15
        # prefix
        ax.plot([0.1, border_len - 0.1], [y_brk, y_brk], color=COLORS["match"], lw=2.5, solid_capstyle="round")
        ax.text(border_len / 2, y_brk - 0.3, f"prefix[{border_len}]", ha="center",
                fontsize=8, color=COLORS["match"], fontweight="bold")
        # suffix
        suf_start = j - border_len + 1
        ax.plot([suf_start + 0.1, j + 0.8], [y_brk, y_brk], color=COLORS["skip"], lw=2.5, solid_capstyle="round")
        ax.text((suf_start + j + 1) / 2, y_brk - 0.3, f"suffix[{border_len}]", ha="center",
                fontsize=8, color="#1565C0", fontweight="bold")
        # border label
        ax.text(m / 2, -0.9, f"border = \"{P[:border_len]}\"  =>  sp[{j}] = {border_len}",
                ha="center", fontsize=10, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="#E8F5E9", edgecolor=COLORS["match"]))
    else:
        ax.text(m / 2, -0.3, f"no border  =>  sp[{j}] = 0",
                ha="center", fontsize=10, color="#777")

    # fallback info
    if snap["fallbacks"]:
        fb_text = " -> ".join([f"k={old}->{new}" for old, new in snap["fallbacks"]])
        ax.text(m / 2, -1.3, f"fallbacks: {fb_text}", ha="center", fontsize=8,
                color=COLORS["border"],
                bbox=dict(boxstyle="round,pad=0.15", facecolor="#F3E5F5", edgecolor=COLORS["border"], alpha=0.8))

    ax.set_title(f"sp step {step_idx+1}/{len(snapshots)} | j={j} | sp[{j}]={sp[j]}", fontsize=10, pad=6)

    # ── Panel 2: sp array built so far ──
    ax2 = axes[1]
    ax2.set_xlim(-2.0, m + 0.5)
    ax2.set_ylim(-0.3, 2.2)
    ax2.set_aspect("equal")
    ax2.axis("off")

    for i in range(m):
        ax2.text(i + 0.45, 1.85, P[i], ha="center", fontsize=10, fontfamily="monospace",
                 color="#333" if i <= j else "#CCC")

    for i in range(m):
        computed = i <= j
        if i == j:
            color = COLORS["current"]
        elif computed and sp[i] > 0:
            color = "#E8D5F5"
        elif computed:
            color = "#FAFAFA"
        else:
            color = "#F0F0F0"
        rect = mpatches.FancyBboxPatch(
            (i, 0.3), 0.9, 0.9, boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="#555" if computed else "#CCC",
            linewidth=1.0 if computed else 0.5)
        ax2.add_patch(rect)
        val = str(sp[i]) if computed else "?"
        ax2.text(i + 0.45, 0.75, val, ha="center", va="center", fontsize=12,
                 fontweight="bold" if computed else "normal",
                 color="#333" if computed else "#BBB")
        ax2.text(i + 0.45, 0.05, str(i), ha="center", fontsize=7, color="#999")
    ax2.text(-0.3, 0.75, "sp", ha="right", va="center", fontsize=11, color="#555")

    safe_tight_layout()
    plt.show()


P_sp_input = widgets.Text(value="abcabcab", description="P:", layout=widgets.Layout(width="400px"))
step_sp = widgets.IntSlider(value=0, min=0, max=0, description="Step:", layout=widgets.Layout(display="none"))


def _update_sp_max(*_):
    P = P_sp_input.value
    if P and len(P) >= 2:
        step_sp.max = max(len(sp_trace(P)) - 1, 0)
        step_sp.value = min(step_sp.value, step_sp.max)

P_sp_input.observe(_update_sp_max, "value")
_update_sp_max()


def _draw_sp(P, step):
    if P and len(P) >= 2:
        draw_sp_step(P, step)

out_sp = widgets.interactive_output(_draw_sp, {"P": P_sp_input, "step": step_sp})
stepper_sp = make_stepper(step_sp, "Step")
display(P_sp_input, stepper_sp, out_sp)

Text(value='abcabcab', description='P:', layout=Layout(width='400px'))

Output()

### KMP Search — Step by Step

Watch how KMP uses the failure function to avoid re-scanning characters. The arrow shows where `j` falls back to on a mismatch.

In [3]:
def kmp_trace(T, P):
    """Return list of snapshots with full state."""
    sp = compute_sp(P)
    n, m = len(T), len(P)
    snapshots = []
    j = 0
    total = 0
    matches = []
    for i in range(n):
        while j > 0 and T[i] != P[j]:
            total += 1
            old_j = j
            j = sp[j - 1]
            snapshots.append({
                "i": i, "j": old_j, "new_j": j, "event": "fallback",
                "align": i - old_j, "new_align": i - j,
                "total_comps": total, "matches": list(matches),
                "sp_used": old_j - 1,
            })
        total += 1
        if T[i] == P[j]:
            snapshots.append({
                "i": i, "j": j, "new_j": j, "event": "match_char",
                "align": i - j, "new_align": i - j,
                "total_comps": total, "matches": list(matches),
            })
            j += 1
            if j == m:
                matches.append(i - m + 1)
                snapshots.append({
                    "i": i, "j": j, "new_j": sp[j - 1], "event": "match_found",
                    "align": i - m + 1, "new_align": i - sp[j - 1] + 1,
                    "total_comps": total, "matches": list(matches),
                    "sp_used": j - 1,
                })
                j = sp[j - 1]
        else:
            snapshots.append({
                "i": i, "j": j, "new_j": j, "event": "mismatch",
                "align": i - j, "new_align": i - j,
                "total_comps": total, "matches": list(matches),
            })
    return snapshots


def draw_kmp_step(T, P, step_idx):
    sp = compute_sp(P)
    snapshots = kmp_trace(T, P)
    if not snapshots:
        return
    step_idx = min(step_idx, len(snapshots) - 1)
    snap = snapshots[step_idx]
    n, m = len(T), len(P)
    i, j, new_j = snap["i"], snap["j"], snap["new_j"]
    event = snap["event"]
    align = snap["align"]

    fig = plt.figure(figsize=(max(n * 0.85, 10), 8.5))
    gs = fig.add_gridspec(3, 1, height_ratios=[3.5, 2.5, 2.5], hspace=0.35)

    # ── Panel 1: T and P alignment ──
    ax = fig.add_subplot(gs[0])
    ax.set_xlim(-2.5, n + 2.0)
    ax.set_ylim(-1.2, 4.0)
    ax.set_aspect("equal")
    ax.axis("off")

    # index row
    for idx in range(n):
        ax.text(idx + 0.45, 3.5, str(idx), ha="center", fontsize=8, color="#999")

    # T row
    t_hi = {}
    if event == "match_char":
        t_hi[i] = COLORS["match"]
    elif event == "mismatch":
        t_hi[i] = COLORS["mismatch"]
    elif event == "fallback":
        t_hi[i] = COLORS["current"]
    elif event == "match_found":
        for k in range(m):
            t_hi[align + k] = COLORS["match"]
    draw_string_row(ax, 2.3, T, label="T", highlights=t_hi)

    # i pointer — above T row
    ax.annotate(f"i={i}", xy=(i + 0.45, 3.2), xytext=(i + 0.45, 3.7),
                fontsize=9, ha="center", color="#1565C0", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#1565C0", lw=1.5))

    # P row
    p_hi = {}
    if event == "match_char":
        p_hi[j] = COLORS["match"]
    elif event == "mismatch":
        p_hi[j] = COLORS["mismatch"]
    elif event == "fallback":
        p_hi[j] = COLORS["mismatch"]
        if new_j < j:
            p_hi[new_j] = COLORS["current"]
    elif event == "match_found":
        for k in range(m):
            p_hi[k] = COLORS["match"]
    draw_string_row(ax, 0.7, P, label="P", highlights=p_hi, offset=align)

    # j pointer — below P row
    j_display = j if event != "match_found" else m - 1
    ax.annotate(f"j={j}", xy=(align + j_display + 0.45, 0.7),
                xytext=(align + j_display + 0.45, 0.1),
                fontsize=9, ha="center", color="#C62828", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#C62828", lw=1.5))

    # fallback arrow — curved arc below the P row
    if event == "fallback" and new_j != j:
        ax.annotate("",
                     xy=(align + new_j + 0.45, 0.65),
                     xytext=(align + j + 0.45, 0.65),
                     arrowprops=dict(arrowstyle="-|>", color=COLORS["border"],
                                     lw=2.5, connectionstyle="arc3,rad=-0.4"))
        mid_x = align + (j + new_j) / 2 + 0.45
        ax.text(mid_x, -0.6,
                f"sp[{j-1}]={new_j}", ha="center", fontsize=10,
                color=COLORS["border"], fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                          edgecolor=COLORS["border"], alpha=0.9))

    # event description
    if event == "fallback":
        info = f"FALLBACK: j={j} -> j=sp[{j-1}]={new_j} (T[{i}]='{T[i]}' != P[{j}]='{P[j]}')"
    elif event == "match_found":
        info = f"FULL MATCH at position {align}! Then j -> sp[{m-1}]={new_j}"
    elif event == "match_char":
        info = f"MATCH: T[{i}]=P[{j}]='{T[i]}', j -> {j+1}"
    else:
        info = f"MISMATCH: T[{i}]='{T[i]}' != P[{j}]='{P[j]}', j stays 0"

    ax.set_title(f"KMP step {step_idx+1}/{len(snapshots)} | comps={snap['total_comps']} | "
                 f"matches={snap['matches']}\n{info}", fontsize=10, pad=8)

    # ── Panel 2: sp table with highlight ──
    ax2 = fig.add_subplot(gs[1])
    ax2.set_xlim(-2.0, m + 0.5)
    ax2.set_ylim(-0.5, 2.5)
    ax2.set_aspect("equal")
    ax2.axis("off")
    ax2.set_title("Failure function sp[] (longest proper border of P[0..j])",
                  fontsize=10, pad=5, loc="left")

    for idx in range(m):
        ax2.text(idx + 0.45, 2.2, str(idx), ha="center", fontsize=8, color="#999")

    draw_string_row(ax2, 1.2, P, label="P")

    sp_used = snap.get("sp_used", None)
    for idx in range(m):
        if sp_used is not None and idx == sp_used:
            color = COLORS["current"]
        elif sp[idx] > 0:
            color = "#E8D5F5"
        else:
            color = "#F5F5F5"
        rect = mpatches.FancyBboxPatch(
            (idx, 0.0), 0.9, 0.9,
            boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="#555", linewidth=1.0)
        ax2.add_patch(rect)
        ax2.text(idx + 0.45, 0.45, str(sp[idx]),
                 ha="center", va="center", fontsize=12, fontweight="bold")
    ax2.text(-0.3, 0.45, "sp", ha="right", va="center", fontsize=11, color="#555")

    if sp_used is not None and 0 <= sp_used < m and sp[sp_used] > 0:
        blen = sp[sp_used]
        ax2.plot([0.1, blen - 0.1], [-0.25, -0.25], color=COLORS["match"], lw=2.5)
        ax2.text(blen / 2, -0.45, "prefix", ha="center", fontsize=8, color=COLORS["match"])
        start = sp_used - blen + 2
        ax2.plot([start - 0.1, sp_used + 0.9], [-0.25, -0.25], color=COLORS["skip"], lw=2.5)
        ax2.text((start + sp_used + 1) / 2, -0.45, "border", ha="center", fontsize=8, color="#1565C0")

    # ── Panel 3: progress timeline ──
    ax3 = fig.add_subplot(gs[2])
    ax3.set_xlim(-0.5, len(snapshots))
    ax3.set_ylim(-0.3, 1.2)
    ax3.axis("off")
    ax3.set_title("Timeline (green=match, red=mismatch, yellow=fallback, star=full match)",
                  fontsize=9, pad=3, loc="left")

    event_colors = {
        "match_char": COLORS["match"],
        "mismatch": COLORS["mismatch"],
        "fallback": COLORS["current"],
        "match_found": "#2E7D32",
    }
    for idx, sn in enumerate(snapshots):
        c = event_colors.get(sn["event"], "#DDD")
        alpha = 1.0 if idx <= step_idx else 0.25
        marker = "*" if sn["event"] == "match_found" else "s"
        size = 80 if sn["event"] == "match_found" else 30
        ax3.scatter(idx, 0.5, c=c, s=size, marker=marker, alpha=alpha, edgecolors="#555", linewidth=0.5)
        if idx == step_idx:
            ax3.scatter(idx, 0.5, c="none", s=120, marker="o", edgecolors="#000", linewidth=2)

    safe_tight_layout(fig)
    plt.show()


# --- Widgets using interactive_output for reliable updates ---
T_kmp = widgets.Text(value="abcaabcabcaab", description="T:", layout=widgets.Layout(width="400px"))
P_kmp = widgets.Text(value="abcab", description="P:", layout=widgets.Layout(width="400px"))
step_kmp = widgets.IntSlider(value=0, min=0, max=1, description="Step:", continuous_update=True)


def _update_kmp_max(*_):
    """Keep slider max in sync with the number of KMP snapshots."""
    T, P = T_kmp.value, P_kmp.value
    if T and P and len(P) <= len(T):
        new_max = max(len(kmp_trace(T, P)) - 1, 0)
        step_kmp.max = new_max


T_kmp.observe(_update_kmp_max, "value")
P_kmp.observe(_update_kmp_max, "value")
_update_kmp_max()


def _draw_kmp(T, P, step):
    if T and P and len(P) <= len(T):
        draw_kmp_step(T, P, step)


out_kmp = widgets.interactive_output(_draw_kmp, {"T": T_kmp, "P": P_kmp, "step": step_kmp})
stepper_kmp = make_stepper(step_kmp, "Step")
display(T_kmp, P_kmp, stepper_kmp, out_kmp)

Text(value='abcaabcabcaab', description='T:', layout=Layout(width='400px'))

Text(value='abcab', description='P:', layout=Layout(width='400px'))

Output()